# Assignment 31: Conversational Chatbot
---

# PART 1 — PDF Ingestion & Preprocessing
## Task 1: Load PDF Documents

In [37]:
from dotenv import load_dotenv

In [38]:
load_dotenv()

True

In [39]:
from langchain_openai import ChatOpenAI

In [40]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("OpenAI LLM initialized successfully.")

OpenAI LLM initialized successfully.


In [41]:
from langchain_community.document_loaders import PyPDFLoader

In [42]:
loader = PyPDFLoader("document.pdf")

documents = loader.load()

In [43]:
print("Number of pages:", len(documents))

Number of pages: 1


In [44]:
documents[0].page_content[:1500]

'Dummy GenAI Knowledge Document\nGenerative AI is a branch of artificial intelligence that can create new content such as text, code, images, and\nsummaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context\nprovided to them.\nRetrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from\nexternal documents before generating an answer. A typical RAG pipeline loads documents, splits them into\nchunks, creates embeddings, stores the embeddings in a vector store, retrieves relevant chunks, and sends\nthe retrieved context to the language model.\nPrompt templates make LLM applications easier to maintain. Instead of hard-coding every question, a\ntemplate can contain placeholders such as {question}. The application fills those placeholders dynamically at\nruntime.\nChat prompt templates are useful for conversational applications because they separate system instructions,\nhuman messages, and optional AI messag

## Task 2: Text Splitting

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [46]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [47]:
chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 3


In [48]:
print(chunks[0].page_content)

Dummy GenAI Knowledge Document
Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and
summaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context
provided to them.
Retrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from
external documents before generating an answer. A typical RAG pipeline loads documents, splits them into


In [49]:
print(chunks[0].metadata)

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-13T11:47:10+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-13T11:47:10+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'document.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


# PART 2 — Embeddings & Vector Store
## Task 3: Create Embeddings

In [50]:
from langchain_huggingface import HuggingFaceEmbeddings

In [51]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4407.49it/s]


In [52]:
embedding = embeddings.embed_query(
    "What is the main topic of this PDF?"
)

print("Embedding size:", len(embedding))

Embedding size: 384


## Task 4: Vector Store Setup

In [53]:
from langchain_community.vectorstores import FAISS

In [54]:
vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector store created successfully.")

Vector store created successfully.


In [55]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [56]:
query = "What is the main topic of this document?"

results = retriever.invoke(query)

In [57]:

print("Retrieved chunks:", len(results))

Retrieved chunks: 3


In [58]:
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i + 1} ---")
    print(doc.page_content)


--- Chunk 1 ---
human messages, and optional AI messages. This structure makes the intended conversation flow explicit.
Example rule: Answer questions only from the supplied context. If the answer is not present in the context,
say that the information is not available in the provided document.

--- Chunk 2 ---
Dummy GenAI Knowledge Document
Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and
summaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context
provided to them.
Retrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from
external documents before generating an answer. A typical RAG pipeline loads documents, splits them into

--- Chunk 3 ---
chunks, creates embeddings, stores the embeddings in a vector store, retrieves relevant chunks, and sends
the retrieved context to the language model.
Prompt templates make LLM applications easier t

# PART 3 — Conversational Prompt with Message History
## Task 5: RAG Prompt Template

In [59]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

In [60]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an AI PDF assistant.

Answer the user's question using ONLY the retrieved PDF context.

Use the conversation history only to understand follow-up questions.

Do not use outside knowledge.

If the answer cannot be found in the retrieved PDF context,
say exactly: "I don't know."

Retrieved PDF context:
{context}
"""
    ),

    MessagesPlaceholder(
        variable_name="chat_history"
    ),

    ("human", "{question}")
])

In [61]:
from langchain_core.messages import HumanMessage, AIMessage

In [62]:
chat_history = []

In [63]:
def ask_pdf(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    response = llm.invoke(messages)
    chat_history.append(
        HumanMessage(content=question)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )

    return response.content

In [64]:
ask_pdf("What is the main topic of this document?")

'The main topic of the document is Generative AI, specifically focusing on its ability to create new content and the mechanisms involved in Retrieval-Augmented Generation (RAG) for improving large language model applications.'

In [65]:
ask_pdf("Can you explain that in simple words?")

"I don't know."

In [66]:
ask_pdf("What are the important points about it?")

'The important points about the document include:\n\n1. Generative AI is a type of artificial intelligence that creates new content, such as text, code, images, and summaries.\n2. Large Language Models (LLMs) generate text by predicting likely words based on the context given to them.\n3. Retrieval-Augmented Generation (RAG) enhances LLM applications by retrieving relevant information from external documents before generating answers.\n4. The document discusses the process of loading documents, splitting them into chunks, creating embeddings, storing those embeddings, and retrieving relevant chunks for use in generating responses.\n5. Prompt templates are mentioned as a way to make LLM applications easier to maintain by using placeholders for dynamic content during runtime.'

In [67]:
for message in chat_history:
    print(
        type(message).__name__,
        ":",
        message.content
    )

HumanMessage : What is the main topic of this document?
AIMessage : The main topic of the document is Generative AI, specifically focusing on its ability to create new content and the mechanisms involved in Retrieval-Augmented Generation (RAG) for improving large language model applications.
HumanMessage : Can you explain that in simple words?
AIMessage : I don't know.
HumanMessage : What are the important points about it?
AIMessage : The important points about the document include:

1. Generative AI is a type of artificial intelligence that creates new content, such as text, code, images, and summaries.
2. Large Language Models (LLMs) generate text by predicting likely words based on the context given to them.
3. Retrieval-Augmented Generation (RAG) enhances LLM applications by retrieving relevant information from external documents before generating answers.
4. The document discusses the process of loading documents, splitting them into chunks, creating embeddings, storing those embe

In [68]:
ask_pdf("Who is the president of Mars?")

"I don't know."

# PART 4 — Conversational RAG Chain
## Task 6: Build Conversational RAG Chain

In [69]:
def conversational_rag(question):
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })

    response = llm.invoke(messages)
    chat_history.append(
        HumanMessage(content=question)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )

    return response.content

In [70]:
conversational_rag("What is the main topic of this PDF?")

'The main topic of the PDF is Generative AI, focusing on its ability to create new content and the process of Retrieval-Augmented Generation (RAG) to enhance large language model applications.'

In [71]:
conversational_rag(
    "Can you explain that in simple words?"
)

"I don't know."

## Task 7: Maintain Message History

In [72]:
for message in chat_history:
    print(
        type(message).__name__,
        ":",
        message.content
    )

HumanMessage : What is the main topic of this document?
AIMessage : The main topic of the document is Generative AI, specifically focusing on its ability to create new content and the mechanisms involved in Retrieval-Augmented Generation (RAG) for improving large language model applications.
HumanMessage : Can you explain that in simple words?
AIMessage : I don't know.
HumanMessage : What are the important points about it?
AIMessage : The important points about the document include:

1. Generative AI is a type of artificial intelligence that creates new content, such as text, code, images, and summaries.
2. Large Language Models (LLMs) generate text by predicting likely words based on the context given to them.
3. Retrieval-Augmented Generation (RAG) enhances LLM applications by retrieving relevant information from external documents before generating answers.
4. The document discusses the process of loading documents, splitting them into chunks, creating embeddings, storing those embe

## Task 8: Trimming Chat History

In [73]:
MAX_MESSAGES = 6

def trim_history(history):

    if len(history) > MAX_MESSAGES:
        return history[-MAX_MESSAGES:]

    return history

In [74]:
def conversational_rag(question):
    chat_history[:] = trim_history(chat_history)

    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    response = llm.invoke(messages)
    chat_history.append(
        HumanMessage(content=question)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )
    chat_history[:] = trim_history(chat_history)

    return response.content

# PART 5 — Multi-Turn Conversation Testing

## Task 9: Follow-Up Q&A Testing

conversational_rag() function already handles retrieval, history, OpenAI, and trimming.

In [75]:
print(conversational_rag("What is the main topic?"))

The main topic of the PDF is Generative AI, which involves creating new content like text, code, images, and summaries, and discusses how Retrieval-Augmented Generation (RAG) enhances large language model applications.


In [76]:
print(conversational_rag("Explain it in simple words."))

I don't know.


In [77]:
print(conversational_rag("Give me an example."))

An example mentioned in the context is the use of prompt templates in large language model (LLM) applications. These templates can have placeholders like {question}, which the application fills in with specific questions at runtime, making it easier to manage and maintain the application.


In [78]:
print(conversational_rag("What are its important points?"))

The important points from the PDF context are:

1. Generative AI creates new content such as text, code, images, and summaries.
2. Large Language Models (LLMs) generate text by predicting likely tokens based on provided context.
3. Retrieval-Augmented Generation (RAG) enhances LLM applications by retrieving relevant information from external documents before generating answers.
4. Prompt templates help maintain LLM applications by allowing dynamic filling of placeholders at runtime.
5. Chat prompt templates are useful for conversational applications as they structure the conversation flow by separating system instructions, human messages, and optional AI messages.


In [79]:
for message in chat_history:
    print(type(message).__name__,":",message.content)

HumanMessage : Explain it in simple words.
AIMessage : I don't know.
HumanMessage : Give me an example.
AIMessage : An example mentioned in the context is the use of prompt templates in large language model (LLM) applications. These templates can have placeholders like {question}, which the application fills in with specific questions at runtime, making it easier to manage and maintain the application.
HumanMessage : What are its important points?
AIMessage : The important points from the PDF context are:

1. Generative AI creates new content such as text, code, images, and summaries.
2. Large Language Models (LLMs) generate text by predicting likely tokens based on provided context.
3. Retrieval-Augmented Generation (RAG) enhances LLM applications by retrieving relevant information from external documents before generating answers.
4. Prompt templates help maintain LLM applications by allowing dynamic filling of placeholders at runtime.
5. Chat prompt templates are useful for conversa

# PART 6 — Mini Project: Conversational PDF Chatbot
## Task 10: Build Final Chatbot Application

In [80]:
def conversational_rag(question):
    chat_history[:] = trim_history(chat_history)

    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )
    messages = rag_prompt.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    response = llm.invoke(messages)
    chat_history.append(
        HumanMessage(content=question)
    )
    chat_history.append(
        AIMessage(content=response.content)
    )
    chat_history[:] = trim_history(chat_history)

    return response.content

In [81]:
print(conversational_rag("What is the main topic?"))

The main topic of the PDF context is Generative AI, specifically focusing on how it creates content and the role of Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG) in enhancing LLM applications.


In [82]:
print(conversational_rag("Explain it in simple words."))

I don't know.


In [83]:
print(conversational_rag("Give me an example."))

I don't know.


In [84]:
print(conversational_rag("What are its important points?"))

The important points from the PDF context are:

1. Generative AI can create new content such as text, code, images, and summaries.
2. Large Language Models (LLMs) generate text by predicting likely tokens based on the provided context.
3. Retrieval-Augmented Generation (RAG) enhances LLM applications by retrieving relevant information from external documents before generating answers.
4. The process involves loading documents, splitting them into chunks, creating embeddings, storing them in a vector store, and retrieving relevant chunks for the language model.
5. Prompt templates help maintain LLM applications by allowing dynamic filling of placeholders at runtime, making it easier to manage questions and responses.


In [85]:
for message in chat_history:
    print(type(message).__name__,":",message.content)

HumanMessage : Explain it in simple words.
AIMessage : I don't know.
HumanMessage : Give me an example.
AIMessage : I don't know.
HumanMessage : What are its important points?
AIMessage : The important points from the PDF context are:

1. Generative AI can create new content such as text, code, images, and summaries.
2. Large Language Models (LLMs) generate text by predicting likely tokens based on the provided context.
3. Retrieval-Augmented Generation (RAG) enhances LLM applications by retrieving relevant information from external documents before generating answers.
4. The process involves loading documents, splitting them into chunks, creating embeddings, storing them in a vector store, and retrieving relevant chunks for the language model.
5. Prompt templates help maintain LLM applications by allowing dynamic filling of placeholders at runtime, making it easier to manage questions and responses.


# Task 11: Observations & Insights

1. Difference between PDF Q&A and conversational PDF Q&A
- PDF Q&A answers each question using the retrieved PDF content but does not necessarily remember previous turns.
- Conversational PDF Q&A combines PDF retrieval with conversation history, allowing the chatbot to understand follow-up questions.

2. Role of message history in follow-up questions
- Message history provides previous user and AI messages to the model through MessagesPlaceholder.

3. Trade-offs between long memory and performance
Longer history provides more conversational context, but it also increases:
- Token usage
- Prompt size
- Processing time
- API cost
Shorter history reduces these costs but can remove older context required for a follow-up question.

4. How trimming history affects answer quality
- Trimming removes old messages once the history exceeds the configured limit.
- If the required information is in recent messages, the effect is minimal.
- If a new question refers to information from an old removed message, the chatbot may lose that conversational context and produce a less useful answer.